# ComplianceGPT Pipeline — Unified Colab Entrypoints

This notebook is designed for **Google Colab ** execution and produces a **verifier-ready Final Answer Contract**.

**Single source of truth:** `src/compliancegpt/generator/citation_contract_80053.md`


## Run Batch (Gold Sets)

This notebook runs the pipeline over **Rev4 (36Q)** or **Rev5 (100Q)** gold set and saves outputs to Drive.

**Default configuration in this file is Rev4.**


In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os, sys, json
PROJECT_ROOT = "/content/drive/MyDrive/ComplianceGPT"
assert os.path.isdir(PROJECT_ROOT), f"PROJECT_ROOT not found: {PROJECT_ROOT}"

SRC_DIR = os.path.join(PROJECT_ROOT, "src")
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

import compliancegpt
print("compliancegpt.__file__ =", compliancegpt.__file__)


compliancegpt.__file__ = None


In [3]:
# --- Canonical paths (Drive) ---
DATA_DIR = os.path.join(PROJECT_ROOT, "data")
PIPELINE_DIR = os.path.join(PROJECT_ROOT, "src", "compliancegpt", "pipeline")

REV5_CATALOG_PATH = os.path.join(DATA_DIR, "ccs", "nist800-53", "NIST_SP-800-53_rev5_catalog.jsonl")
REV4_CATALOG_PATH = os.path.join(DATA_DIR, "ccs", "nist800-53", "NIST_SP-800-53_rev4_catalog.jsonl")

REV5_GOLD_PATH = os.path.join(DATA_DIR, "gold_standard_datasets", "nist800-53", "nist_sp800-53_rev5_gold-set_100q.csv")
REV4_GOLD_PATH = os.path.join(DATA_DIR, "gold_standard_datasets", "nist800-53", "nist_sp800-53_rev4_gold-set_36q.csv")

QUR_REV5_PATH = os.path.join(DATA_DIR, "qur_outputs", "qur_rewrites_rev5.csv")
QUR_REV4_PATH = os.path.join(DATA_DIR, "qur_outputs", "qur_rewrites_rev4.csv")
QUR_ERROR_BANK_PATH = os.path.join(DATA_DIR, "qur_outputs", "qur_rewrites_error_bank.csv")

ERROR_BANK_PATH = os.path.join(DATA_DIR, "error_bank", "error_bank_v1.csv")

ORG_PROFILE_REV5 = os.path.join(DATA_DIR, "ODP", "rev5", "org_profile_example_rev5.yaml")
ORG_PROFILE_REV4 = os.path.join(DATA_DIR, "ODP", "rev4", "org_profile_example_rev4.yaml")

for p in [REV5_CATALOG_PATH, REV4_CATALOG_PATH, REV5_GOLD_PATH, REV4_GOLD_PATH]:
    if not os.path.exists(p):
        print("[WARN] Missing:", p)


In [4]:
# --- Helpers: load gold sets and QUR rewrite maps ---
from typing import Any
import pandas as pd

def load_gold(framework_version: str) -> pd.DataFrame:
    path = REV5_GOLD_PATH if framework_version.lower() == "rev5" else REV4_GOLD_PATH
    # The gold CSVs are often not UTF-8; use latin1 for robustness.
    df = pd.read_csv(path, encoding="latin1")
    # Standardize columns
    if "id" in df.columns and "query_id" not in df.columns:
        df["query_id"] = df["id"].astype(str)
    if "question" in df.columns and "original_query" not in df.columns:
        df["original_query"] = df["question"].astype(str)
    return df

def load_qur_map(framework_version: str) -> dict[str, list[str]]:
    path = QUR_REV5_PATH if framework_version.lower() == "rev5" else QUR_REV4_PATH
    if not os.path.exists(path):
        print(f"[INFO] QUR CSV not found: {path}")
        return {}
    df = pd.read_csv(path)
    # Expected columns from QUR generator: query_id, rewrite_rank, rewritten_query
    required = {"query_id", "rewrite_rank", "rewritten_query"}
    if not required.issubset(set(df.columns)):
        raise ValueError(f"Unexpected QUR CSV schema. Missing {required - set(df.columns)}. Columns={list(df.columns)}")
    out: dict[str, list[str]] = {}
    for qid, g in df.groupby(df["query_id"].astype(str)):
        g2 = g.sort_values("rewrite_rank")
        out[qid] = [str(x) for x in g2["rewritten_query"].tolist() if isinstance(x, str) and x.strip()]
    return out


In [5]:
# --- Build pipeline (single place to configure knobs) ---
from compliancegpt.pipeline.pipeline import ComplianceGPTPipeline

def build_pipeline(
    *,
    framework_version: str,
    use_qur_live: bool,
    use_org_profile: bool,
    verify_strict_extras: bool = False,
    verify_strict_verbatim: bool = True,
    verify_strict_version: bool = False,
):
    fw = framework_version.lower().strip()
    if fw not in {"rev4", "rev5"}:
        raise ValueError(f"framework_version must be rev4 or rev5 (got {framework_version!r})")

    ccs_path = REV5_CATALOG_PATH if fw == "rev5" else REV4_CATALOG_PATH
    org_profile = (ORG_PROFILE_REV5 if fw == "rev5" else ORG_PROFILE_REV4) if use_org_profile else None
    if org_profile and not os.path.exists(org_profile):
        print(f"[WARN] org_profile_path not found; continuing without profile: {org_profile}")
        org_profile = None

    pipe = ComplianceGPTPipeline(
        framework_version=fw,
        ccs_path=ccs_path,
        use_qur=bool(use_qur_live),
        doc_filter_mode="prefer_smt_keep_params",
        org_profile_path=org_profile,
        verify_strict_extras=bool(verify_strict_extras),
        verify_strict_verbatim=bool(verify_strict_verbatim),
        verify_strict_version=bool(verify_strict_version),
    )
    return pipe


[QUR] RUN_MODE=lib
[QUR] Library mode loaded. (Batch generation & Tests disabled)
QUR library exported as QUR_LIB
[QUR] RUN_MODE=lib


### Configure batch run

In [6]:
# === USER CONFIG ===
FRAMEWORK_VERSION = "rev4"   # "rev4" or "rev5"

# Rewrites
USE_QUR_PRECOMPUTED = True   # use data/qur_outputs/*.csv if available
USE_QUR_LIVE = False         # if True, the pipeline can generate rewrites if none are provided (slower)

# Org profile (only needed for FILL_FROM_PROFILE)
USE_ORG_PROFILE = False

# Verification strictness knobs
VERIFY_STRICT_EXTRAS = False
VERIFY_STRICT_VERBATIM = True
VERIFY_STRICT_VERSION = False

TOP_K = 12
MAX_ROWS = None              # e.g., 20 for quick test; None for full dataset
OUTPUT_DIR = os.path.join(PROJECT_ROOT, "experiments", "pipeline_runs")


In [7]:
# --- Build pipeline ---
pipe = build_pipeline(
    framework_version=FRAMEWORK_VERSION,
    use_qur_live=USE_QUR_LIVE,
    use_org_profile=USE_ORG_PROFILE,
    verify_strict_extras=VERIFY_STRICT_EXTRAS,
    verify_strict_verbatim=VERIFY_STRICT_VERBATIM,
    verify_strict_version=VERIFY_STRICT_VERSION,
)
print("[OK] Pipeline initialized.")

modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/133M [00:00<?, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

[OK] Pipeline initialized.


In [8]:
# --- Load dataset and rewrites ---
gold_df = load_gold(FRAMEWORK_VERSION)
if MAX_ROWS is not None:
    gold_df = gold_df.head(int(MAX_ROWS))

qur_map = load_qur_map(FRAMEWORK_VERSION) if USE_QUR_PRECOMPUTED else {}
print("rows:", len(gold_df), "qur_map_ids:", len(qur_map))


rows: 36 qur_map_ids: 36


In [9]:
from dataclasses import replace

# Align retriever-specific knobs with this notebook's TOP_K, but do not assume every retriever exposes them.
if hasattr(pipe, "retriever") and hasattr(pipe.retriever, "config") and hasattr(pipe.retriever.config, "clauses_per_control"):
    pipe.retriever.config = replace(pipe.retriever.config, clauses_per_control=int(TOP_K))
    print("clauses_per_control =", pipe.retriever.config.clauses_per_control)
else:
    print("[INFO] retriever config does not expose clauses_per_control; skipping.")


clauses_per_control = 12


In [10]:
# --- Run batch ---
from tqdm.auto import tqdm

rows = []

def _gold_field(gold_row: dict, keys: list[str], default=""):
    for k in keys:
        if k in gold_row and gold_row.get(k) not in (None, ""):
            return gold_row.get(k)
    return default

def _stringify_maybe_list(x) -> str:
    if x is None:
        return ""
    if isinstance(x, list):
        return "|".join([str(v) for v in x])
    return str(x)

for _, r in tqdm(gold_df.iterrows(), total=len(gold_df), desc=f"Pipeline batch ({FRAMEWORK_VERSION})"):
    gold_row = dict(r.to_dict())
    qid = str(gold_row.get("query_id", gold_row.get("id", "")))
    query = str(gold_row.get("original_query", gold_row.get("question", ""))).strip()

    rewrites = qur_map.get(qid, []) if USE_QUR_PRECOMPUTED else []

    out = pipe.answer(
        query,
        top_k=int(TOP_K),
        rewrites=rewrites if rewrites else None,
        gold_row=gold_row,
        use_generator=True,
        run_verify=True,
    )
    c = out.get("contract", out)

    rows.append({
        "query_id": qid,
        "framework_version": FRAMEWORK_VERSION,
        "question": query,
"gold_control_id": _stringify_maybe_list(_gold_field(gold_row, ["control_id", "gold_control_id", "control"], default="")),
"gold_source_version": _stringify_maybe_list(_gold_field(gold_row, ["source_version", "gold_source_version", "framework_version"], default="")),
"gold_resolution_policy": _stringify_maybe_list(_gold_field(gold_row, ["resolution_policy", "gold_resolution_policy"], default="")),
"gold_control_path": _stringify_maybe_list(_gold_field(gold_row, ["gold_control_path", "control_path", "doc_ids", "doc_id_list", "gold_doc_ids"], default="")),

        "status": c.get("status"),
        "verifier_pass": bool(c.get("verifier_pass")),
        "verifier_errors": "|".join(list(c.get("verifier_errors") or [])),
        "primary_citation": c.get("primary_citation", ""),
        "all_citations": c.get("all_citations", ""),
        "odp_required_list": "|".join(list(c.get("odp_required_list") or [])),
        "selected_source_ids": "|".join(list(c.get("selected_source_ids") or [])),
        # Store full contract JSON for audit/debug
        "contract_json": json.dumps(c, ensure_ascii=False),
    })

df_out = pd.DataFrame(rows)
df_out.head()

Pipeline batch (rev4):   0%|          | 0/36 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


,query_id,framework_version,question,gold_control_id,gold_source_version,gold_resolution_policy,gold_control_path,status,verifier_pass,verifier_errors,primary_citation,all_citations,odp_required_list,selected_source_ids,contract_json
0,1,rev4,What does AC-2 require the organization to do ...,AC-2,Revision 4,ASK,ac-2_smt.f,PARAMS_REQUIRED,True,,ac-2_smt.a,"ac-2_smt.a, ac-2_smt.b, ac-2_smt.d, ac-2_smt.e...",ac-2_prm_1|ac-2_prm_2|ac-2_prm_3|ac-2_prm_4,ac-2_smt.a|ac-2_smt.b|ac-2_smt.d|ac-2_smt.e|ac...,"{""question"": ""What does AC-2 require the organ..."
1,2,rev4,What principle does AC-6 require the organizat...,AC-6,Revision 4,nan,ac-6_smt,OK,True,,ac-6_gdn,"ac-6_smt, ac-6_gdn",,ac-6_smt|ac-6_gdn,"{""question"": ""What principle does AC-6 require..."
2,3,rev4,What security awareness training must the orga...,AT-2,Revision 4,FILL_FROM_PROFILE,at-2_smt\nat-2_smt.a\nat-2_smt.b\nat-2_smt.c,PARAMS_REQUIRED,True,,at-2_smt,"at-2_smt, at-2_smt.a, at-2_smt.b, at-2_smt.c",at-2_prm_1,at-2_smt|at-2_smt.a|at-2_smt.b|at-2_smt.c,"{""question"": ""What security awareness training..."
3,4,rev4,What are the requirements for role-based secur...,AT-3,Revision 4,FILL_FROM_PROFILE,at-3_smt\nat-3_smt.a\nat-3_smt.b\nat-3_smt.c,PARAMS_REQUIRED,False,MissedDocIds:['at-3_smt'],at-3_gdn,"at-3_smt.a, at-3_smt.b, at-3_smt.c, at-3_gdn",at-3_prm_1,at-3_smt.a|at-3_smt.b|at-3_smt.c|at-3_gdn,"{""question"": ""What are the requirements for ro..."
4,5,rev4,What must the organization determine about eve...,AU-2,Revision 4,ASK,au-2_smt.d,PARAMS_REQUIRED,True,,au-2_smt.d,"au-2_smt.d, au-2_smt.a",au-2_prm_1|au-2_prm_2,au-2_smt.d|au-2_smt.a,"{""question"": ""What must the organization deter..."


In [11]:
# --- Save to Drive ---
import os, time
os.makedirs(OUTPUT_DIR, exist_ok=True)
ts = time.strftime("%Y%m%d_%H%M%S")
csv_path = os.path.join(OUTPUT_DIR, f"pipeline_{FRAMEWORK_VERSION}_contracts_{ts}.csv")
df_out.to_csv(csv_path, index=False)
print("[OK] Saved:", csv_path)

# Quick summary
print(df_out["status"].value_counts(dropna=False))
print("verifier_pass_rate:", float(df_out["verifier_pass"].mean()) if len(df_out) else 0.0)


[OK] Saved: /content/drive/MyDrive/ComplianceGPT/experiments/pipeline_runs/pipeline_rev4_contracts_20260312_152654.csv
status
PARAMS_REQUIRED    28
OK                  8
Name: count, dtype: int64
verifier_pass_rate: 0.75


## Multi-metric evaluation report

This section reports multiple evaluation metrics to reduce sensitivity to strict clause-level exact matching.


In [12]:
import json
import numpy as np
import pandas as pd

def _safe_json_load(x):
    if not isinstance(x, str) or not x.strip():
        return {}
    try:
        return json.loads(x)
    except Exception:
        return {}

contracts = df_out["contract_json"].apply(_safe_json_load)

# Flatten verifier_metrics (gold-dependent) and basic contract fields (gold-independent)
verifier_metrics = contracts.apply(lambda c: c.get("verifier_metrics") or {})
vm_df = pd.json_normalize(verifier_metrics).add_prefix("vm.")

df_eval = df_out.copy()
df_eval["evidence_span_count"] = contracts.apply(lambda c: len(c.get("evidence_spans") or []))
df_eval["has_evidence_spans"] = df_eval["evidence_span_count"] > 0

df_eval = pd.concat([df_eval, vm_df], axis=1)

# Robust helpers for gold metadata (if present)
def _count_gold_clause_ids(x) -> int:
    if x is None:
        return 0
    if isinstance(x, float) and np.isnan(x):
        return 0
    if isinstance(x, list):
        return len([v for v in x if str(v).strip()])
    s = str(x).strip()
    if not s or s.lower() == "nan":
        return 0
    # If serialized JSON list
    if s.startswith("[") and s.endswith("]"):
        try:
            arr = json.loads(s)
            if isinstance(arr, list):
                return len([v for v in arr if str(v).strip()])
        except Exception:
            pass
    # Otherwise treat as pipe-delimited list
    return len([t for t in s.split("|") if t.strip()])

df_eval["gold_clause_count"] = df_eval.get("gold_control_path", "").apply(_count_gold_clause_ids) if "gold_control_path" in df_eval.columns else 0

# Derived indicators (verifier_metrics may be missing for ERROR or exception cases)
def _to_float(x):
    try:
        return float(x)
    except Exception:
        return float("nan")

df_eval["control_recall"] = df_eval.get("vm.control_recall", np.nan).apply(_to_float) if "vm.control_recall" in df_eval.columns else float("nan")
df_eval["doc_recall"] = df_eval.get("vm.doc_recall", np.nan).apply(_to_float) if "vm.doc_recall" in df_eval.columns else float("nan")
df_eval["verbatim_strict_pass_rate"] = df_eval.get("vm.verbatim_strict_pass_rate", np.nan).apply(_to_float) if "vm.verbatim_strict_pass_rate" in df_eval.columns else float("nan")
df_eval["verbatim_normalized_pass_rate"] = df_eval.get("vm.verbatim_normalized_pass_rate", np.nan).apply(_to_float) if "vm.verbatim_normalized_pass_rate" in df_eval.columns else float("nan")

df_eval["control_hit_any"] = df_eval["control_recall"].fillna(0.0) > 0.0
df_eval["control_full_recall"] = df_eval["control_recall"].fillna(0.0) >= (1.0 - 1e-9)
df_eval["doc_hit_any"] = df_eval["doc_recall"].fillna(0.0) > 0.0
df_eval["doc_full_recall"] = df_eval["doc_recall"].fillna(0.0) >= (1.0 - 1e-9)

df_eval.head()


,query_id,framework_version,question,gold_control_id,gold_source_version,gold_resolution_policy,gold_control_path,status,verifier_pass,verifier_errors,...,vm.verbatim_normalized_pass_rate,gold_clause_count,control_recall,doc_recall,verbatim_strict_pass_rate,verbatim_normalized_pass_rate,control_hit_any,control_full_recall,doc_hit_any,doc_full_recall
0,1,rev4,What does AC-2 require the organization to do ...,AC-2,Revision 4,ASK,ac-2_smt.f,PARAMS_REQUIRED,True,,...,1.0,1,1.0,1.00,1.0,1.0,True,True,True,True
1,2,rev4,What principle does AC-6 require the organizat...,AC-6,Revision 4,nan,ac-6_smt,OK,True,,...,1.0,1,1.0,1.00,1.0,1.0,True,True,True,True
2,3,rev4,What security awareness training must the orga...,AT-2,Revision 4,FILL_FROM_PROFILE,at-2_smt\nat-2_smt.a\nat-2_smt.b\nat-2_smt.c,PARAMS_REQUIRED,True,,...,1.0,1,1.0,1.00,1.0,1.0,True,True,True,True
3,4,rev4,What are the requirements for role-based secur...,AT-3,Revision 4,FILL_FROM_PROFILE,at-3_smt\nat-3_smt.a\nat-3_smt.b\nat-3_smt.c,PARAMS_REQUIRED,False,MissedDocIds:['at-3_smt'],...,1.0,1,1.0,0.75,1.0,1.0,True,True,True,False
4,5,rev4,What must the organization determine about eve...,AU-2,Revision 4,ASK,au-2_smt.d,PARAMS_REQUIRED,True,,...,1.0,1,1.0,1.00,1.0,1.0,True,True,True,True


In [13]:
import numpy as np
import pandas as pd

def _rate(s: pd.Series) -> float:
    if len(s) == 0:
        return 0.0
    return float(np.mean(s.astype(bool)))

def _nanmean(s: pd.Series) -> float:
    if len(s) == 0:
        return float("nan")
    return float(np.nanmean(pd.to_numeric(s, errors="coerce").values))

overall = pd.DataFrame([{
    "n": int(len(df_eval)),
    "strict_verifier_pass_rate": _rate(df_eval["verifier_pass"]),
    "control_hit_rate": _rate(df_eval["control_hit_any"]),
    "control_full_recall_rate": _rate(df_eval["control_full_recall"]),
    "doc_hit_rate": _rate(df_eval["doc_hit_any"]),
    "doc_full_recall_rate": _rate(df_eval["doc_full_recall"]),
    "avg_evidence_span_count": float(df_eval["evidence_span_count"].mean()) if len(df_eval) else 0.0,
    "has_evidence_spans_rate": _rate(df_eval["has_evidence_spans"]),
    "avg_verbatim_strict_pass_rate": _nanmean(df_eval["verbatim_strict_pass_rate"]),
    "avg_verbatim_normalized_pass_rate": _nanmean(df_eval["verbatim_normalized_pass_rate"]),
}])

display(overall)

# Breakdown by contract status
by_status = df_eval.groupby("status", dropna=False).agg(
    n=("query_id", "count"),
    strict_verifier_pass_rate=("verifier_pass", _rate),
    control_hit_rate=("control_hit_any", _rate),
    doc_hit_rate=("doc_hit_any", _rate),
    avg_evidence_span_count=("evidence_span_count", "mean"),
).reset_index()

display(by_status)

# Breakdown by gold resolution policy, if present
if "gold_resolution_policy" in df_eval.columns:
    by_policy = df_eval.groupby("gold_resolution_policy", dropna=False).agg(
        n=("query_id", "count"),
        strict_verifier_pass_rate=("verifier_pass", _rate),
        control_hit_rate=("control_hit_any", _rate),
        doc_hit_rate=("doc_hit_any", _rate),
    ).reset_index()
    display(by_policy)

# Top error tags
from collections import Counter

tags = []
for s in df_eval["verifier_errors"].fillna("").astype(str).tolist():
    if not s.strip():
        continue
    tags.extend([t.strip() for t in s.split("|") if t.strip()])
top_tags = pd.DataFrame(Counter(tags).most_common(30), columns=["error_tag", "count"])
display(top_tags)

# Relationship: strict pass rate vs. gold clause count buckets (if available)
if "gold_clause_count" in df_eval.columns:
    def _bucket(n: int) -> str:
        if n <= 1:
            return "0-1"
        if n == 2:
            return "2"
        if n in (3, 4):
            return "3-4"
        return "5+"
    df_eval["gold_clause_count_bucket"] = df_eval["gold_clause_count"].apply(lambda x: _bucket(int(x)))
    bucket_table = df_eval.groupby("gold_clause_count_bucket", dropna=False).agg(
        n=("query_id", "count"),
        strict_verifier_pass_rate=("verifier_pass", _rate),
        control_hit_rate=("control_hit_any", _rate),
        doc_hit_rate=("doc_hit_any", _rate),
        avg_gold_clause_count=("gold_clause_count", "mean"),
    ).reset_index().sort_values("gold_clause_count_bucket")
    display(bucket_table)


,n,strict_verifier_pass_rate,control_hit_rate,control_full_recall_rate,doc_hit_rate,doc_full_recall_rate,avg_evidence_span_count,has_evidence_spans_rate,avg_verbatim_strict_pass_rate,avg_verbatim_normalized_pass_rate
0,36,0.75,1.0,1.0,0.972222,0.75,4.222222,1.0,1.0,1.0


,status,n,strict_verifier_pass_rate,control_hit_rate,doc_hit_rate,avg_evidence_span_count
0,OK,8,0.75,1.0,1.000000,2.875000
1,PARAMS_REQUIRED,28,0.75,1.0,0.964286,4.607143


,gold_resolution_policy,n,strict_verifier_pass_rate,control_hit_rate,doc_hit_rate
0,ASK,5,0.800000,1.0,1.000000
1,FILL_FROM_PROFILE,12,0.750000,1.0,1.000000
2,PRESERVE,2,0.500000,1.0,1.000000
3,nan,17,0.764706,1.0,0.941176


,error_tag,count
0,MissedDocIds:['at-3_smt'],1
1,MissedDocIds:['ia-5_gdn'],1
2,MissedDocIds:['ir-6_smt.b'],1
3,PreserveMissingPlaceholder:pe-03_prm.01,1
4,PreserveMissingPlaceholder:pe-03_prm.04,1
5,PreserveMissingPlaceholder:pe-03_prm.06,1
6,PreserveMissingPlaceholder:pe-03_prm.07,1
7,PreserveMissingPlaceholder:pe-03_prm.08,1
8,PreserveMissingPlaceholder:pe-03_prm.09,1
9,ODPRequiredListNotEqualGold:gold=['pe-03_prm.0...,1


,gold_clause_count_bucket,n,strict_verifier_pass_rate,control_hit_rate,doc_hit_rate,avg_gold_clause_count
0,0-1,36,0.75,1.0,0.972222,1.0


In [14]:
import os

reports_dir = os.path.join(OUTPUT_DIR, "reports")
os.makedirs(reports_dir, exist_ok=True)

report_csv = os.path.join(reports_dir, f"eval_{FRAMEWORK_VERSION}_{ts}.csv")
df_eval.to_csv(report_csv, index=False)

def _to_markdown_safe(df: pd.DataFrame) -> str:
    try:
        return df.to_markdown(index=False)
    except Exception:
        return df.to_string(index=False)

md_lines = []
md_lines.append(f"# ComplianceGPT Batch Evaluation Report ({FRAMEWORK_VERSION})")
md_lines.append("")
md_lines.append(f"- Timestamp: {ts}")
md_lines.append(f"- Dataset rows: {len(df_eval)}")
md_lines.append("")

md_lines.append("## Overall")
md_lines.append("")
md_lines.append(_to_markdown_safe(overall))
md_lines.append("")

md_lines.append("## By status")
md_lines.append("")
md_lines.append(_to_markdown_safe(by_status))
md_lines.append("")

if "gold_resolution_policy" in df_eval.columns:
    md_lines.append("## By gold resolution policy")
    md_lines.append("")
    try:
        md_lines.append(_to_markdown_safe(by_policy))
    except Exception:
        md_lines.append("(No policy breakdown available.)")
    md_lines.append("")

md_lines.append("## Top verifier error tags")
md_lines.append("")
md_lines.append(_to_markdown_safe(top_tags))
md_lines.append("")

if "gold_clause_count_bucket" in df_eval.columns:
    md_lines.append("## Strict pass rate vs. gold clause count")
    md_lines.append("")
    md_lines.append(_to_markdown_safe(bucket_table))
    md_lines.append("")

report_md = os.path.join(reports_dir, f"eval_{FRAMEWORK_VERSION}_{ts}.md")
with open(report_md, "w", encoding="utf-8") as f:
    f.write("\n".join(md_lines))

print("[OK] Saved evaluation artifacts:")
print(" -", report_csv)
print(" -", report_md)


[OK] Saved evaluation artifacts:
 - /content/drive/MyDrive/ComplianceGPT/experiments/pipeline_runs/reports/eval_rev4_20260312_152654.csv
 - /content/drive/MyDrive/ComplianceGPT/experiments/pipeline_runs/reports/eval_rev4_20260312_152654.md
